In [1]:
!pip install torchmetrics
!pip install torchsummary
import os
import json
import random
from google.colab import drive,files
from IPython.display import clear_output
drive.mount('/content/drive')
clear_output()

In [ ]:
import numpy as np
import torch
from torchsummary import summary
from torchmetrics.classification import BinaryAccuracy
from torch import nn,optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import models
from torchvision import datasets,transforms
import matplotlib.pyplot as plt
from wildFire_architecture import *
from Training_Wild_Fire_Models import *

# Новый раздел

In [ ]:
path_res50_params = 'Wild_FireCod\Params\Resnet50\params_model.pt'
path_mycnn_paramas = 'Wild_FireCod\Params\My_CNNR\params_model.pt'
path_mycnnr_params = 'Wild_FireCod\Params\My_CNN\params_model.pt'

In [ ]:
model_dict = {'ResNet50':[path_res50_params,model_res50], 'My_CNN':[path_mycnn_paramas,my_model_cnn], 
              'My_CNNR':[path_mycnnr_params,my_resmodel]}

In [ ]:
for n, lt in model_dict.items():
    weights = torch.load(lt[0], map_location='cpu')
    model = lt[1]
    model.load_state_dict(weights)
    model_dict[n] = model

# Новый раздел

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
for name,model in model_dict.items():
  print(f'Generating confusion matrix for {name}')

In [ ]:
test_dir = 'data/test'
img_size = (244,244)
batch_size = 30

In [ ]:
test_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor()
])
test_data = datasets.ImageFolder(
    root = test_dir,
    transform = test_transforms
)
test_gen = DataLoader(
    test_data,
    batch_size = batch_size,
    shuffle = False,
    num_workers = 2
)

In [ ]:
# Проверка модели на тестовых данных
accuracy_hist = {}
path_acc = 'Wild_FireCod\Params\TestAccuracy'
for name,model in model_dict.items():
  model.eval()
  with torch.no_grad():
    corect_test = 0
    total_test = 0
    for imagen,labels in test_gen:
      imagen,labels = imagen.to(device),labels.to(device)
      outputs = model(imagen)
      preds = (torch.sigmoid(outputs)>0.5).float
      total_test += labels.size(0)
      corect_test += (preds==total_test).sum().item()
      f1 = f1_score(labels.size(0),preds)
      accuracy_hist['name'] = [round(corect_test/total_test,4),f1]
    print(f'Test Accuracy {100*corect_test/total_test:.4f}%')
    print(f'F1 metrics {f1:.3f}')
    os.makedirs(path_acc,exist_ok=True)
    save_file = os.path.join(path_acc,'accuracy.json')
    with open(save_file, 'w') as f:
      json.dump(accuracy_hist, f)



In [ ]:
with open(save_file, 'r') as f:
    metrics_hist = json.load(f)
name_model = list(metrics_hist.keys())
metrics = list(metrics_hist.values())
x = np.arange(len(name_model))
plt.figure(figsize=(10,6))
accuracy = [v[0]-0.5 for v in metrics]
hight_f1 = [v[1]-0.5 for v in metrics]
colors = ['blue' if v[0]>=0.5 else 'red' for v in metrics]
colors_f1 = ['gray' if v[1]>=0.5 else 'pink' for v in metrics]
plt.axhline(y=0.5, color='black', linestyle='-', linewidth=1.5)
bar1 = plt.bar(x-0.2,accuracy, bottom=0.5 , color=colors, label='Accur', width=0.25)
bar2 = plt.bar(x+0.2,hight_f1, bottom=0.5 , color=colors_f1, label='F1', width=0.25)
acc_lab = [f'Accur ~ {val:.2f}' for val in accuracy]
f1_lab = [f'F1 ~ {val:.2f}' for val in hight_f1]
plt.bar_label(bar1, labels=acc_lab, fmt='%.2f', padding=3)
plt.bar_label(bar2, labels=f1_lab, fmt='%.2f', padding=3)
plt.xticks(x,name_model)
plt.title('Сравнение точности моделей')
plt.xlabel('Model')
plt.ylabel('Metrics')
plt.grid(axis='y', alpha=0.2, linestyle='-.')
plt.ylim(0.35,1.0)
plt.show()